# Run the validation on new data/ from configuration files 
## Step 1: Re-initialise the Data Context from Lakehouse Files

In [ ]:
# Check if Great Expectations rules are in place

import notebookutils

path = "<workspace-id>@onelake.dfs.fabric.microsoft.com/<lakehouse-id>/Files/gx"  

try:
    notebookutils.fs.ls(path)
    exists = True
except Exception:
    exists = False
# if no rules are in place, we run the SetupFile to implement the rules, otherwise we continue with validation
if not exists:
    notebookutils.notebook.run("nb_SetupDataContext_with_GreatExpectations", 3600)
else:
    print("Folder already exists – continue with flow.")

In [5]:
# We get the exisiting data context into the notebook
import great_expectations as gx

path_to_local_context = '/lakehouse/default/Files'

# initialise the data context from the Lakehouse Files
context = gx.get_context(project_root_dir=path_to_local_context)


StatementMeta(, d753c810-d61b-4b16-9a86-7f781ca31c93, 9, Finished, Available, Finished, False)

In [ ]:
import json
from great_expectations.util import convert_to_json_serializable

# Load the cleaned data from the Bronze layer (Parquet, written by nb_DataWrangling) that is to be validated
bronze_df_validation = spark.read.parquet("Files/bronze/cleaned_data")

# Pass the Spark DataFrame to Great Expectations as the batch to validate
batch_parameters = {"dataframe": bronze_df_validation}

# Get the checkpoint created in nb_SetupDataContext_with_GreatExpectations
# (it bundles the expectation suite and the validation definition)
checkpoint = context.checkpoints.get("gx_checkpoint_1")

# Derive a readable identifier from the OneLake path (last path segment = table name "CustomerChurn")
file_identifier = "abfss://<workspace-id>@onelake.dfs.fabric.microsoft.com/<lakehouse-id>/Tables/bronze/CustomerChurn".split("/")[-1]

# Give this validation run a name so it can be identified in the results / audit trail
runid = gx.RunIdentifier(run_name=f"Test run - {file_identifier}")

# Run the validation of the batch against all expectations in the checkpoint
results = checkpoint.run(batch_parameters=batch_parameters, run_id=runid)

# Create a summary of the validation results (success flag, passed/failed expectations)
json_string = results.describe()

# Print the summary; used to decide whether the data may be loaded into the Silver layer
print(json_string)

## Step 2: Handle results
Now we have some results, we want to perform some actions: 
- log the results in a validation log Lakehouse table
- if success, write to Silver Table
- if failure, throw an exception to be handled by data pipeline 

In [ ]:
from datetime import datetime
import pandas as pd

def load_to_silver(validated_bronze):
    """Write the validated Bronze DataFrame to the Silver layer as a Delta table."""
    # Overwrite the Silver table with the latest validated data
    validated_bronze.write.format("delta").mode("overwrite").saveAsTable("silver.customerchurn")


def parse_and_load_checkpoint_result(results):
    """Extract key metrics from the GX checkpoint result, store them in a
    Delta audit table and return the overall validation status (True/False)."""

    # A checkpoint can contain several validations; take the first validation result
    validation_result = next(iter(results.run_results.values()))

    # Extract statistics and metadata directly from the result objects
    success = results.success              # overall pass/fail of the checkpoint
    stats = validation_result.statistics   # counts and percentages of expectations
    meta = validation_result.meta          # metadata such as the expectation suite name

    # Get run name and run time; fall back to defaults if no run ID is available
    run_name = results.run_id.run_name if results.run_id else "N/A"
    run_time_str = results.run_id.run_time if results.run_id else datetime.now().isoformat()

    # Parse the timestamp (ISO format, cut to seconds) into a datetime object
    parsed_time = datetime.strptime(str(run_time_str)[:19], "%Y-%m-%d %H:%M:%S")

    # Flatten the results into a single-row structure for the audit table
    restructured = {
        "run_name": [run_name],
        "run_time": [parsed_time],
        "validation_result": [success],
        "evaluated_expectations": [stats.get("evaluated_expectations", 0)],
        "successful_expectations": [stats.get("successful_expectations", 0)],
        "unsuccessful_expectations": [stats.get("unsuccessful_expectations", 0)],
        "success_percent": [stats.get("success_percent", 0.0)],
        "expectation_suite_name": [meta.get("expectation_suite_name", "N/A")]
    }

    # Convert to a Spark DataFrame and append to the validation history table
    # (append keeps a full audit trail of all runs, e.g. for ISO 42001)
    pandas_df = pd.DataFrame(restructured)
    spark_df = spark.createDataFrame(pandas_df)
    spark_df.write.format("delta").mode("append").save("Tables/validation_results")

    return success

# --- Run PIPELINE ---

# 1. Run the checkpoint on the Bronze data (returns a GX 1.x CheckpointResult object)
results = checkpoint.run(batch_parameters={"dataframe": bronze_df_validation})

# 2. Store the results in the audit table and get the overall success flag
success = parse_and_load_checkpoint_result(results)

# 3. Branch on the validation result: only load to Silver if all checks passed
if success:
    print("✅ Validation successful! Loading data to Silver layer ...")
    load_to_silver(bronze_df_validation)
else:
    print("❌ Validation failed!")
    # Raise an exception so the pipeline activity fails and triggers the failure notification
    raise Exception("Data validation failed. Handle the exception in the data pipeline.")